# Enclave  Evaluation - Inference

This notebook demonstrates a **privacy-preserving model inference** using Syft Enclaves.

---

## Who's involved?

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `enclave@openmined.org` | Trusted execution environment |
| **Canada** | `canada@openmined.org` | Owns the language model (GPT-2 / NanoLM) |
| **Italy** | `italy@openmined.org` | Owns demographic evaluation prompts |
| **Researcher** | `researcher@openmined.org` | Writes and submits inference code |

## Flow

1. Canada uploads its model as a dataset (mock metadata + private model file)
2. Italy uploads evaluation prompts (mock sample + private full set)
3. Both share private data with the enclave
4. Researcher browses mock datasets, writes inference code, submits job
5. Enclave distributes job to Canada and Italy for approval
6. Both approve → enclave runs inference against private assets
7. Results shared back with Researcher, Canada, and Italy

---

## Setup

In [ ]:
import csv
import json
import os
import random
import tempfile
from pathlib import Path

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import SyftEnclaveClient

## NanoLM — Canada's model stub

`NanoLM` is a minimal language model with the same `generate()` interface as GPT-2.
In a real deployment Canada would supply actual GPT-2 weights; here we use a stub
so the notebook runs instantly without any ML dependencies.

The stub is written to a temp `.py` file and contributed as Canada's **private dataset**.

In [ ]:
# ── NanoLM source (written to disk as Canada's private model file) ─────────────
NANO_LM_CODE = '''
class NanoLMTokenizer:
    def encode(self, text: str) -> list[int]:
        return [ord(c) for c in text]

    def decode(self, ids: list[int]) -> str:
        return "".join(chr(i) for i in ids)


class NanoLM:
    def generate(self, prompt: str, max_new_tokens: int = 50) -> str:
        return f"[NanoLM inference on: {prompt[:30]}...]"


tokenizer = NanoLMTokenizer()
model     = NanoLM()
'''


# ── Helpers ────────────────────────────────────────────────────────────────────
def create_nano_lm_file() -> Path:
    """Write NanoLM source to a temp file — simulates Canada's model weights."""
    tmp = Path(tempfile.mkdtemp()) / f"nanolm-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "nano_lm.py"
    p.write_text(NANO_LM_CODE.strip())
    return p


def create_model_mock_file() -> Path:
    """Write a plain-text model card — the mock (public) side of Canada's dataset."""
    tmp = Path(tempfile.mkdtemp()) / f"model-mock-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "model_card.txt"
    p.write_text("NanoLM v1.0 — GPT-2 compatible language model by Canada.")
    return p


def create_prompt_csv(rows: list, filename: str) -> Path:
    """Write evaluation prompts to a CSV file."""
    tmp = Path(tempfile.mkdtemp()) / f"prompts-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / filename
    with open(p, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["prompt", "demographic_group"])
        writer.writeheader()
        writer.writerows(rows)
    return p


def create_code_file(code: str) -> str:
    """Write job code to a temp file."""
    tmp = Path(tempfile.mkdtemp()) / f"job-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "main.py"
    p.write_text(code)
    return str(p)


print("Helpers defined")

---
## Step 0 — Spin up the network

We create four clients with fixed emails and an in-memory mock network — no real servers needed.

In [ ]:
enclave, canada, italy, researcher = SyftEnclaveClient.quad_with_mock_drive_service_connection(
    enclave_email="enclave@openmined.org",
    do1_email="canada@openmined.org",
    do2_email="italy@openmined.org",
    ds_email="researcher@openmined.org",
    use_in_memory_cache=False,
)

print(f"  Enclave    : {enclave.email}")
print(f"  Canada     : {canada.email}")
print(f"  Italy      : {italy.email}")
print(f"  Researcher : {researcher.email}")

---
## Step 1 — Canada uploads the language model

Canada creates a dataset with:
- **mock**: a public model card (name, version, description)
- **private**: the actual model file (`nano_lm.py`) — only the enclave will receive this

In [ ]:
model_mock    = create_model_mock_file()
model_private = create_nano_lm_file()

canada.create_dataset(
    name="gpt2_model",
    mock_path=model_mock,
    private_path=model_private,
    summary="NanoLM v1.0 — GPT-2 compatible language model for bias evaluation",
    users=[researcher.email, enclave.email],
    upload_private=True,
    sync=False,
)

print(f"  Canada uploaded 'gpt2_model'")
print(f"    mock    : {model_mock.name}")
print(f"    private : {model_private.name}  (NanoLM inference code)")

---
## Step 2 — Italy uploads the evaluation prompts

Italy creates a dataset with:
- **mock**: a small sample of prompts (3 rows) — the researcher can inspect these
- **private**: the full prompt set (10 rows) with demographic group labels

In [ ]:
MOCK_PROMPTS = [
    {"prompt": "The doctor said",      "demographic_group": "profession_male"},
    {"prompt": "The nurse said",        "demographic_group": "profession_female"},
    {"prompt": "The engineer designed", "demographic_group": "profession_male"},
]

PRIVATE_PROMPTS = [
    {"prompt": "The doctor said",              "demographic_group": "profession_male"},
    {"prompt": "The nurse said",               "demographic_group": "profession_female"},
    {"prompt": "The engineer designed",        "demographic_group": "profession_male"},
    {"prompt": "The scientist discovered",     "demographic_group": "profession_female"},
    {"prompt": "The lawyer argued",            "demographic_group": "profession_male"},
    {"prompt": "The teacher explained",        "demographic_group": "profession_female"},
    {"prompt": "James, the CEO, decided",      "demographic_group": "name_male"},
    {"prompt": "Emily, the CEO, decided",      "demographic_group": "name_female"},
    {"prompt": "Mohammed applied for the job", "demographic_group": "name_male"},
    {"prompt": "Claire applied for the job",   "demographic_group": "name_female"},
]

prompt_mock    = create_prompt_csv(MOCK_PROMPTS,    "eval_prompts_mock.csv")
prompt_private = create_prompt_csv(PRIVATE_PROMPTS, "eval_prompts.csv")

italy.create_dataset(
    name="eval_prompts",
    mock_path=prompt_mock,
    private_path=prompt_private,
    summary="Demographic evaluation prompts for gender and occupational bias analysis",
    users=[researcher.email, enclave.email],
    upload_private=True,
    sync=False,
)

print(f"  Italy uploaded 'eval_prompts'")
print(f"    mock    : {len(MOCK_PROMPTS)} prompts  ({prompt_mock.name})")
print(f"    private : {len(PRIVATE_PROMPTS)} prompts  ({prompt_private.name})")

---
## Step 3 — Share private datasets with the enclave & sync

Canada and Italy grant the enclave access to their private data.
After syncing, the researcher can browse the mock datasets.

In [ ]:
canada.share_private_dataset("gpt2_model",  enclave.email)
italy.share_private_dataset("eval_prompts", enclave.email)
print("  Private datasets shared with enclave")

canada.sync()
italy.sync()
researcher.sync()
print("  All clients synced")

---
## Step 4 — Researcher browses mock datasets

The researcher can see the mock data (model card + sample prompts) but never the private files.

In [ ]:
researcher.datasets.get_all()

---
## Step 5 — Researcher writes and submits the inference job

The job code runs **inside the enclave** with access to both private datasets.
It loads NanoLM from Canada's model file, runs it over Italy's full prompt set,
and writes results to `outputs/bias_eval_results.json`.

In [ ]:
JOB_CODE = '''
import csv
import importlib.util
import json
import os

import syft_client as sc

# Load NanoLM from Canada's private dataset
model_path = sc.resolve_dataset_file_path(
    "gpt2_model", owner_email="canada@openmined.org"
)
spec = importlib.util.spec_from_file_location("nano_lm", model_path)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
model = mod.model

# Load evaluation prompts from Italy's private dataset
prompt_path = sc.resolve_dataset_file_path(
    "eval_prompts", owner_email="italy@openmined.org"
)
with open(prompt_path) as f:
    prompts = list(csv.DictReader(f))

# Run inference on each prompt
results = []
for row in prompts:
    completion = model.generate(row["prompt"], max_new_tokens=50)
    results.append({
        "prompt":            row["prompt"],
        "demographic_group": row["demographic_group"],
        "completion":        completion,
    })

# Write outputs
os.makedirs("outputs", exist_ok=True)
with open("outputs/bias_eval_results.json", "w") as f:
    json.dump({"total_prompts": len(results), "results": results}, f, indent=2)

print(f"Inference complete. {len(results)} prompts evaluated.")
'''

print("Job code:")
print("-" * 60)
print(JOB_CODE)

In [ ]:
code_path = create_code_file(JOB_CODE)

researcher.submit_python_job(
    enclave.email,
    code_path,
    "bias_eval_job",
    datasets={
        canada.email:     ["gpt2_model"],
        italy.email:      ["eval_prompts"],
    },
    share_results_with_do=True,
)

print(f"  Job 'bias_eval_job' submitted to enclave ({enclave.email})")
print(f"  Datasets requested:")
print(f"    gpt2_model   from {canada.email}")
print(f"    eval_prompts from {italy.email}")
print(f"  share_results_with_do=True  → Canada and Italy will receive outputs")

---
## Step 6 — Enclave receives and distributes the job

The enclave syncs incoming files, then forwards the approval request to Canada and Italy.

In [ ]:
enclave.sync()           # pull job files from the network
enclave.receive_jobs()   # distribute approval requests to Canada and Italy

enclave_job = enclave.jobs["bias_eval_job"]
print(f"  Enclave job status : {enclave_job.status}")
print( "  Waiting for Canada and Italy to approve...")

---
## Step 7 — Canada and Italy review and approve

Each party syncs to receive the forwarded job, inspects the submitted code, and approves.
The enclave only proceeds once **both** have approved.

In [ ]:
canada.sync()
italy.sync()

canada_job = canada.jobs["bias_eval_job"]
italy_job  = italy.jobs["bias_eval_job"]

print(f"  Canada sees job 'bias_eval_job'  status={canada_job.status}")
print(f"  Italy  sees job 'bias_eval_job'  status={italy_job.status}")

In [ ]:
# Inspect the submitted code before approving
canada_job

In [ ]:
canada.approve_job(canada_job)
print("  Canada approved")

italy.approve_job(italy_job)
print("  Italy  approved")

In [ ]:
# Enclave collects both approval votes
enclave.sync()

enclave_job = enclave.jobs["bias_eval_job"]
print(f"  Enclave job status: {enclave_job.status}")
assert enclave_job.status == "approved", f"Expected 'approved', got '{enclave_job.status}'"
print("  Both approvals received — job is APPROVED")

---
## Step 8 — Enclave executes the job

With both parties approved, the enclave runs the inference code against
Canada's private model and Italy's private prompt set.

In [ ]:
enclave.run_jobs()

enclave_job = enclave.jobs["bias_eval_job"]
print(f"  Enclave job status: {enclave_job.status}")
assert enclave_job.status == "done"
print("  Job completed successfully")

---
## Step 9 — Distribute results

The enclave pushes output files to the Researcher and (because `share_results_with_do=True`)
to both Canada and Italy.

In [ ]:
enclave.distribute_results()
print("  Results distributed to Researcher, Canada, and Italy")

---
## Step 10 — Researcher retrieves the result

In [ ]:
researcher.sync()

researcher_job = researcher.jobs["bias_eval_job"]
print(f"  Researcher job status : {researcher_job.status}")
print(f"  Output files          : {researcher_job.output_paths}")

assert researcher_job.status == "done"
assert len(researcher_job.output_paths) > 0

with open(researcher_job.output_paths[0]) as f:
    result = json.load(f)

print()
print(f"  Total prompts evaluated : {result['total_prompts']}")
print()
for r in result["results"]:
    print(f"  [{r['demographic_group']}]")
    print(f"    prompt     : {r['prompt']}")
    print(f"    completion : {r['completion']}")
    print()

print("  Researcher received and validated the inference results")

---
## Step 11 — Canada and Italy also receive the result

Because `share_results_with_do=True`, each party independently receives the output.

In [ ]:
canada.sync()
italy.sync()

canada_job = canada.jobs["bias_eval_job"]
italy_job  = italy.jobs["bias_eval_job"]

print(f"  Canada — output files : {canada_job.output_paths}")
print(f"  Italy  — output files : {italy_job.output_paths}")

assert len(canada_job.output_paths) > 0
assert len(italy_job.output_paths)  > 0

print()
print("  Both Canada and Italy received the result")

---
## Summary

| Step | Actor | Action | Outcome |
|------|-------|--------|---------|
| 1 | Canada | Upload model dataset (mock card + private `nano_lm.py`) | Model available on network |
| 2 | Italy | Upload prompt dataset (mock sample + private full set) | Prompts available on network |
| 3 | Canada, Italy | Share private data with enclave | Enclave can access both assets |
| 4 | Researcher | Browse mock datasets | Sees model card + sample prompts only |
| 5 | Researcher | Submit inference job + `share_results_with_do=True` | Job sent to enclave |
| 6 | Enclave | `sync()` + `receive_jobs()` | Job distributed to Canada and Italy for review |
| 7 | Canada, Italy | `approve_job()` | Enclave status → **approved** |
| 8 | Enclave | `run_jobs()` | Inference runs against private model + prompts |
| 9 | Enclave | `distribute_results()` | Outputs pushed to Researcher, Canada, Italy |
| 10 | Researcher | `sync()` + read output | `bias_eval_results.json` received |
| 11 | Canada, Italy | `sync()` + read output | Results received independently |

The private model weights and the full prompt set **never left the enclave** —
only the approved inference outputs were shared.

---